In [1]:
!pip install xgboost

In [2]:
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import torch

from import_data import select_data

In [3]:
train_dataset, test_dataset = select_data("transformer", "px_py_pz_eta_rel_phi_rel_32")

X_train = train_dataset.tensors[0].numpy()
y_train = train_dataset.tensors[1].numpy()

X_test = test_dataset.tensors[0].numpy()
y_test = test_dataset.tensors[1].numpy()

batch_size, num_constituents, num_features = X_train.shape
X_train = X_train.reshape(batch_size, -1) 
X_test = X_test.reshape(X_test.shape[0], -1)

y_train = np.argmax(y_train, axis=1)
y_test = np.argmax(y_test, axis=1)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/c/users/bonvi/Documents/deepset/data/jetid_px_py_pz_eta_rel_phi_rel_32/processed/all_augmented_features_32.npy'

In [ ]:
# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'max_depth': 10,
    'eta': 0.3,
    'objective': 'multi:softprob',
    'num_class': 5,  
    'eval_metric': 'mlogloss'
}

In [ ]:
num_round = 500
model = xgb.train(params, dtrain, num_round)

preds = model.predict(dtest)
best_preds = np.argmax(preds, axis=1)

In [ ]:
accuracy = accuracy_score(y_test, best_preds)
print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(classification_report(y_test, best_preds))

importance = model.get_score(importance_type='weight')
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 important features:")
for feat, importance in sorted_importance[:10]:
    constituent_idx = int(feat.split('f')[1]) // num_features
    feature_idx = int(feat.split('f')[1]) % num_features
    print(f"Constituent {constituent_idx}, Feature {feature_idx}: {importance}")

Accuracy: 0.70

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.65      0.66     52404
           1       0.70      0.65      0.67     50468
           2       0.67      0.73      0.70     52235
           3       0.69      0.67      0.68     52298
           4       0.77      0.79      0.78     52595

    accuracy                           0.70    260000
   macro avg       0.70      0.70      0.70    260000
weighted avg       0.70      0.70      0.70    260000


Top 10 important features:
Constituent 0, Feature 4: 17379.0
Constituent 0, Feature 3: 17327.0
Constituent 1, Feature 4: 15636.0
Constituent 1, Feature 3: 15362.0
Constituent 2, Feature 4: 14399.0
Constituent 2, Feature 3: 14310.0
Constituent 3, Feature 3: 14005.0
Constituent 3, Feature 4: 13960.0
Constituent 4, Feature 3: 13878.0
Constituent 4, Feature 4: 13769.0
